# The Simplification Tax — Single-Notebook Run

CS 5788 final project — three-way comparison of **PPO, DPO, and SimPO** on TL;DR summarisation, with Pythia-1B-deduped + LoRA and a GPT-4o pairwise judge.

## How to run (3 steps)

1. **Open in Google Colab**: `File → Open notebook → Upload`.
2. **Set runtime to GPU**: `Runtime → Change runtime type → GPU`. Use **A100** if Colab Pro is available, **T4** if free tier (set `SMALL_MODE = True` two cells down).
3. **Set the `OPENAI_API_KEY`** in the first code cell (one line, marked clearly), then `Runtime → Run all`.

That's it. The notebook installs every dependency, downloads data + models, trains all three methods, runs the GPT-4o pairwise eval, plots results, and saves a CSV summary.

## What it produces

Saved under `./results/<timestamp>/`:
- `sft/`, `dpo/default/`, `simpo/default/`, `ppo/default/` — checkpoints + per-step training history
- `eval/summary.csv` — length-controlled win rate, diversity, length, etc. for each method vs SFT
- `eval/<method>.json` — full per-prompt judgments and generations
- `plots/` — training curves, win-rate bars, length bars

## Time budget (single GPU)

| Stage | A100 80GB | T4 (`SMALL_MODE=True`) |
|---|---|---|
| SFT     | ~45 min | ~8 min |
| DPO     | ~25 min | ~5 min |
| SimPO   | ~25 min | ~5 min |
| PPO     | ~60 min | ~12 min |
| Eval    | ~15 min | ~10 min (OpenAI rate-limited) |
| **Total** | **~3 hr** | **~40 min** |

`SMALL_MODE = True` swaps Pythia-1B → Pythia-160m and shrinks the data slice — useful for verifying the pipeline runs end-to-end on free Colab. Headline results in the report come from the full Pythia-1B run on A100.

## Cost

GPT-4o judge: ~400 API calls per method × 3 methods = **~1,200 calls ≈ \$6–\$8**. Set your own `OPENAI_API_KEY` in the first code cell.


## 0. Setup

In [ ]:
import os, sys, subprocess

# ============================================================
# >>> SET YOUR OPENAI API KEY HERE <<<
# Get one at https://platform.openai.com/api-keys (~$8 for the full run).
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# ============================================================

# Toggle: set True on a small/free GPU (e.g. Colab free T4) for a
# fast end-to-end smoke test on Pythia-160m. Set False for the real
# Pythia-1B run on an A100.
SMALL_MODE = False

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

for pkg in ['transformers>=4.40', 'datasets', 'peft>=0.10', 'accelerate',
            'openai>=1.0', 'trl>=0.7.10', 'matplotlib']:
    name = pkg.split('>=')[0].replace('-', '_')
    try:
        __import__(name)
    except ImportError:
        pip(pkg)

import json, math, random, copy, time
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'SMALL_MODE: {SMALL_MODE}')

torch.manual_seed(42); random.seed(42); np.random.seed(42)
RESULTS = './results'
os.makedirs(RESULTS, exist_ok=True)
TS = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'timestamp: {TS}')
BF16 = device == 'cuda'
DTYPE = torch.bfloat16 if BF16 else torch.float32


## 1. Hyperparameters

In [ ]:
if SMALL_MODE:
    # Fast end-to-end smoke (Colab free T4 friendly)
    MODEL_NAME = 'EleutherAI/pythia-160m'
    MAX_LEN  = 256
    N_SFT    = 1024
    N_PREF   = 1024
    N_EVAL   = 100
    N_JUDGE  = 40
    SFT_BATCH  = 8
    PREF_BATCH = 4
else:
    # Real run (Pythia-1B-deduped, A100 recommended)
    MODEL_NAME = 'EleutherAI/pythia-1b-deduped'
    MAX_LEN  = 384
    N_SFT    = 4096
    N_PREF   = 4096
    N_EVAL   = 500
    N_JUDGE  = 200
    SFT_BATCH  = 8
    PREF_BATCH = 4

# Optimisation
SFT_LR    = 2e-5
PREF_LR   = 5e-6

# DPO
DPO_BETA  = 0.1
# SimPO
SIMPO_BETA  = 2.0
SIMPO_GAMMA = 1.0
# PPO
PPO_KL = 0.05
PPO_LR = 1.4e-5

# LoRA
LORA_R     = 16
LORA_ALPHA = 32

print(f'model: {MODEL_NAME}')
print(f'data: n_sft={N_SFT}  n_pref={N_PREF}  n_eval={N_EVAL}  n_judge={N_JUDGE}')


## 2. Data loaders (inline)

In [ ]:
from datasets import load_dataset

def _norm_sft(row):
    prompt = row.get('prompt') or row.get('post')
    response = row.get('completion') or row.get('label') or row.get('summary')
    if not prompt or not response: return None
    p = str(prompt).strip()
    if 'TL;DR' not in p: p = p + '\nTL;DR:'
    return {'prompt': p + ' ', 'response': str(response).strip()}

def _norm_pref(row):
    prompt = row.get('prompt') or row.get('post')
    chosen = row.get('chosen') or row.get('response_chosen')
    rejected = row.get('rejected') or row.get('response_rejected')
    if not (prompt and chosen and rejected): return None
    p = str(prompt).strip()
    if 'TL;DR' not in p: p = p + '\nTL;DR:'
    return {'prompt': p + ' ', 'chosen': str(chosen).strip(), 'rejected': str(rejected).strip()}

print('loading TL;DR SFT...')
sft_raw = load_dataset('trl-lib/tldr', split=f'train[:{N_SFT}]')
sft_data = [x for x in (_norm_sft(r) for r in sft_raw) if x]
print(f'  {len(sft_data)} SFT examples')

print('loading TL;DR preferences...')
pref_raw = load_dataset('CarperAI/openai_summarize_comparisons', split=f'train[:{N_PREF}]')
pref_data = [x for x in (_norm_pref(r) for r in pref_raw) if x]
print(f'  {len(pref_data)} preference pairs')

# Held-out eval slice
eval_pref_raw = load_dataset('CarperAI/openai_summarize_comparisons', split=f'train[{N_PREF}:{N_PREF + N_EVAL}]')
eval_prompts = [r['prompt'].strip() + ('\nTL;DR: ' if 'TL;DR' not in r['prompt'] else ' ') for r in eval_pref_raw]
eval_posts = [p.split('\nTL;DR:')[0] for p in eval_prompts]
print(f'  {len(eval_prompts)} eval prompts')

## 3. Tokenizer + LoRA wrapper

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
PAD = tokenizer.pad_token_id

def fresh_lora_model():
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE).to(device)
    cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
        target_modules=['query_key_value', 'dense'],
        bias='none', task_type=TaskType.CAUSAL_LM,
    )
    return get_peft_model(base, cfg)

def load_lora_model(adapter_path):
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE).to(device)
    return PeftModel.from_pretrained(base, adapter_path, is_trainable=True).to(device)

def trainable_count(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return trainable, total

## 4. Datasets + collate

In [ ]:
class SFTDataset(Dataset):
    def __init__(self, exs): self.exs = exs
    def __len__(self): return len(self.exs)
    def __getitem__(self, i):
        e = self.exs[i]
        p = tokenizer.encode(e['prompt'], add_special_tokens=False)
        r = tokenizer.encode(e['response'] + tokenizer.eos_token, add_special_tokens=False)
        ids = (p + r)[:MAX_LEN]
        mask = ([0]*min(len(p), len(ids)) + [1]*max(0, len(ids)-len(p)))[:MAX_LEN]
        return {'ids': torch.tensor(ids), 'rmask': torch.tensor(mask)}

def sft_collate(batch):
    L = max(len(b['ids']) for b in batch); B = len(batch)
    ids = torch.full((B, L), PAD, dtype=torch.long)
    attn = torch.zeros((B, L), dtype=torch.long)
    rmask = torch.zeros((B, L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b['ids'])
        ids[i, :n] = b['ids']; attn[i, :n] = 1; rmask[i, :n] = b['rmask']
    return {'input_ids': ids, 'attention_mask': attn, 'response_mask': rmask}

class PrefDataset(Dataset):
    def __init__(self, exs): self.exs = exs
    def __len__(self): return len(self.exs)
    def _enc(self, prompt, resp):
        p = tokenizer.encode(prompt, add_special_tokens=False)
        r = tokenizer.encode(resp + tokenizer.eos_token, add_special_tokens=False)
        ids = (p + r)[:MAX_LEN]
        mask = ([0]*min(len(p), len(ids)) + [1]*max(0, len(ids)-len(p)))[:MAX_LEN]
        return ids, mask
    def __getitem__(self, i):
        e = self.exs[i]
        iw, mw = self._enc(e['prompt'], e['chosen'])
        il, ml = self._enc(e['prompt'], e['rejected'])
        return {'iw': torch.tensor(iw), 'mw': torch.tensor(mw),
                'il': torch.tensor(il), 'ml': torch.tensor(ml)}

def pref_collate(batch):
    def pad(ids_l, mask_l):
        L = max(len(s) for s in ids_l); B = len(ids_l)
        ids = torch.full((B, L), PAD, dtype=torch.long)
        attn = torch.zeros((B, L), dtype=torch.long)
        rmask = torch.zeros((B, L), dtype=torch.long)
        for i, (s, m) in enumerate(zip(ids_l, mask_l)):
            n = len(s); ids[i, :n] = s; attn[i, :n] = 1; rmask[i, :n] = m
        return ids, attn, rmask
    iw, aw, rw = pad([b['iw'] for b in batch], [b['mw'] for b in batch])
    il, al, rl = pad([b['il'] for b in batch], [b['ml'] for b in batch])
    return {'iw': iw, 'aw': aw, 'rw': rw, 'il': il, 'al': al, 'rl': rl}

## 5. Loss functions (inline)

In [ ]:
def get_logp_response(model, input_ids, attn, rmask):
    """Sum log p(y_t|x,y_<t) over response tokens. Returns (B,)."""
    out = model(input_ids=input_ids, attention_mask=attn)
    logits = out.logits[:, :-1, :].contiguous()
    labels = input_ids[:, 1:].contiguous()
    mask = rmask[:, 1:].contiguous().float()
    logp = F.log_softmax(logits.float(), dim=-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    return (logp * mask).sum(-1)

def get_avg_logp_response(model, input_ids, attn, rmask):
    """Average log p over response tokens (for SimPO). Returns (sum, n, avg)."""
    out = model(input_ids=input_ids, attention_mask=attn)
    logits = out.logits[:, :-1, :].contiguous()
    labels = input_ids[:, 1:].contiguous()
    mask = rmask[:, 1:].contiguous().float()
    logp = F.log_softmax(logits.float(), dim=-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    logp = logp * mask
    s = logp.sum(-1)
    n = mask.sum(-1).clamp(min=1.0)
    return s, n, s / n

def dpo_loss(logp_w, logp_l, ref_w, ref_l, beta):
    chosen_lr = logp_w - ref_w
    rejected_lr = logp_l - ref_l
    logits = beta * (chosen_lr - rejected_lr)
    loss = -F.logsigmoid(logits).mean()
    rc = beta * chosen_lr.detach()
    rr = beta * rejected_lr.detach()
    return loss, rc.mean(), rr.mean(), (rc > rr).float().mean()

def simpo_loss(avg_w, avg_l, beta, gamma):
    rc = beta * avg_w
    rr = beta * avg_l
    logits = (rc - rr) - gamma
    loss = -F.logsigmoid(logits).mean()
    acc = (rc > rr).float().mean()
    margin = (rc - rr).mean()
    return loss, rc.mean().detach(), rr.mean().detach(), acc.detach(), margin.detach()

## 6. SFT training
Pythia-1B + LoRA on TL;DR. ~45 min on A100.

In [ ]:
SFT_DIR = f'{RESULTS}/sft/{TS}'
os.makedirs(SFT_DIR, exist_ok=True)

sft_model = fresh_lora_model()
tr, tot = trainable_count(sft_model)
print(f'trainable={tr/1e6:.2f}M total={tot/1e6:.1f}M')

sft_loader = DataLoader(SFTDataset(sft_data), batch_size=SFT_BATCH, shuffle=True,
                        collate_fn=sft_collate, num_workers=0)
opt = optim.AdamW([p for p in sft_model.parameters() if p.requires_grad],
                  lr=SFT_LR, betas=(0.9, 0.95), weight_decay=0.01)

sft_history = []
sft_model.train()
step = 0
for batch in sft_loader:
    ids = batch['input_ids'].to(device)
    attn = batch['attention_mask'].to(device)
    rmask = batch['response_mask'].to(device)
    out = sft_model(input_ids=ids, attention_mask=attn)
    sl = out.logits[:, :-1, :].contiguous().float()
    sla = ids[:, 1:].contiguous()
    smk = rmask[:, 1:].contiguous().float()
    lpt = F.cross_entropy(sl.view(-1, sl.size(-1)), sla.view(-1), reduction='none').view(sla.shape)
    loss = (lpt * smk).sum() / smk.sum().clamp(min=1.0)
    opt.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in sft_model.parameters() if p.requires_grad], 1.0)
    opt.step()
    if step % 20 == 0:
        print(f'[sft] step={step:4d} loss={loss.item():.4f}')
        sft_history.append({'step': step, 'loss': float(loss.item())})
    step += 1

sft_model.save_pretrained(f'{SFT_DIR}/model')
tokenizer.save_pretrained(f'{SFT_DIR}/model')
with open(f'{SFT_DIR}/history.json', 'w') as f:
    json.dump(sft_history, f, indent=2)
print('SFT done.')
del sft_model; torch.cuda.empty_cache() if device == 'cuda' else None

## 7. DPO training
Loads the SFT checkpoint as both policy and reference; DPO trains policy.

In [ ]:
DPO_DIR = f'{RESULTS}/dpo/default/{TS}'
os.makedirs(DPO_DIR, exist_ok=True)

dpo_policy = load_lora_model(f'{SFT_DIR}/model')
dpo_ref = load_lora_model(f'{SFT_DIR}/model')
for p in dpo_ref.parameters(): p.requires_grad_(False)
dpo_ref.eval()

pref_loader = DataLoader(PrefDataset(pref_data), batch_size=PREF_BATCH, shuffle=True,
                         collate_fn=pref_collate, num_workers=0)
opt = optim.AdamW([p for p in dpo_policy.parameters() if p.requires_grad],
                  lr=PREF_LR, betas=(0.9, 0.95), weight_decay=0.01)

dpo_history = []
dpo_policy.train()
step = 0
for batch in pref_loader:
    iw = batch['iw'].to(device); aw = batch['aw'].to(device); rw = batch['rw'].to(device)
    il = batch['il'].to(device); al = batch['al'].to(device); rl = batch['rl'].to(device)
    lpw = get_logp_response(dpo_policy, iw, aw, rw)
    lpl = get_logp_response(dpo_policy, il, al, rl)
    with torch.no_grad():
        rew = get_logp_response(dpo_ref, iw, aw, rw)
        rel = get_logp_response(dpo_ref, il, al, rl)
    loss, rc, rr, acc = dpo_loss(lpw, lpl, rew, rel, beta=DPO_BETA)
    opt.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in dpo_policy.parameters() if p.requires_grad], 1.0)
    opt.step()
    if step % 20 == 0:
        len_w = float(rw.sum(-1).float().mean().item())
        len_l = float(rl.sum(-1).float().mean().item())
        print(f'[dpo] step={step:4d} loss={loss.item():.4f} acc={acc.item():.3f} rc={rc.item():+.3f} rr={rr.item():+.3f}')
        dpo_history.append({'step': step, 'loss': float(loss.item()), 'acc': float(acc.item()),
                            'r_chosen': float(rc.item()), 'r_rejected': float(rr.item()),
                            'len_chosen': len_w, 'len_rejected': len_l})
    step += 1

dpo_policy.save_pretrained(f'{DPO_DIR}/model')
tokenizer.save_pretrained(f'{DPO_DIR}/model')
with open(f'{DPO_DIR}/history.json', 'w') as f:
    json.dump(dpo_history, f, indent=2)
print('DPO done.')
del dpo_policy, dpo_ref; torch.cuda.empty_cache() if device == 'cuda' else None

## 8. SimPO training
No reference. Length-normalised reward.

In [ ]:
SIMPO_DIR = f'{RESULTS}/simpo/default/{TS}'
os.makedirs(SIMPO_DIR, exist_ok=True)

simpo_policy = load_lora_model(f'{SFT_DIR}/model')

pref_loader = DataLoader(PrefDataset(pref_data), batch_size=PREF_BATCH, shuffle=True,
                         collate_fn=pref_collate, num_workers=0)
opt = optim.AdamW([p for p in simpo_policy.parameters() if p.requires_grad],
                  lr=PREF_LR, betas=(0.9, 0.95), weight_decay=0.01)

simpo_history = []
simpo_policy.train()
step = 0
for batch in pref_loader:
    iw = batch['iw'].to(device); aw = batch['aw'].to(device); rw = batch['rw'].to(device)
    il = batch['il'].to(device); al = batch['al'].to(device); rl = batch['rl'].to(device)
    _, _, av_w = get_avg_logp_response(simpo_policy, iw, aw, rw)
    _, _, av_l = get_avg_logp_response(simpo_policy, il, al, rl)
    loss, rc, rr, acc, margin = simpo_loss(av_w, av_l, beta=SIMPO_BETA, gamma=SIMPO_GAMMA)
    opt.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in simpo_policy.parameters() if p.requires_grad], 1.0)
    opt.step()
    if step % 20 == 0:
        len_w = float(rw.sum(-1).float().mean().item())
        len_l = float(rl.sum(-1).float().mean().item())
        print(f'[simpo] step={step:4d} loss={loss.item():.4f} acc={acc.item():.3f} rc={rc.item():+.3f} rr={rr.item():+.3f} m={margin.item():+.3f}')
        simpo_history.append({'step': step, 'loss': float(loss.item()), 'acc': float(acc.item()),
                              'r_chosen': float(rc.item()), 'r_rejected': float(rr.item()),
                              'margin': float(margin.item()),
                              'len_chosen': len_w, 'len_rejected': len_l})
    step += 1

simpo_policy.save_pretrained(f'{SIMPO_DIR}/model')
tokenizer.save_pretrained(f'{SIMPO_DIR}/model')
with open(f'{SIMPO_DIR}/history.json', 'w') as f:
    json.dump(simpo_history, f, indent=2)
print('SimPO done.')
del simpo_policy; torch.cuda.empty_cache() if device == 'cuda' else None

## 9. PPO via TRL (optional — slowest)

Uses TRL's `PPOTrainer`. ~60 min on A100. Skip this cell if you want a 2-way comparison (DPO vs SimPO) instead of 3-way.

In [ ]:
RUN_PPO = True  # set False to skip

if RUN_PPO:
    PPO_DIR = f'{RESULTS}/ppo/default/{TS}'
    os.makedirs(PPO_DIR, exist_ok=True)
    try:
        from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
        from transformers import AutoModelForSequenceClassification
        # Reward model
        REWARDS = ['cleanrl/EleutherAI_pythia-1b-deduped__reward__tldr',
                   'OpenAssistant/reward-model-deberta-v3-large-v2']
        reward_model = None
        for rn in REWARDS:
            try:
                reward_tok = AutoTokenizer.from_pretrained(rn)
                reward_model = AutoModelForSequenceClassification.from_pretrained(rn).to(device).eval()
                print(f'[reward] loaded {rn}')
                break
            except Exception as e:
                print(f'[reward] tried {rn}: {e}')
        if reward_model is None:
            raise RuntimeError('no reward model loadable')

        # Merge SFT LoRA -> base for value-head wrapping
        sft_loaded = load_lora_model(f'{SFT_DIR}/model')
        try:
            merged = sft_loaded.merge_and_unload()
        except Exception:
            merged = sft_loaded
        # Fresh LoRA + value head
        cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                         target_modules=['query_key_value','dense'],
                         bias='none', task_type=TaskType.CAUSAL_LM)
        merged = get_peft_model(merged, cfg)
        ppo_policy = AutoModelForCausalLMWithValueHead.from_pretrained(merged)

        ppo_cfg = PPOConfig(model_name=MODEL_NAME, learning_rate=PPO_LR,
                            batch_size=8, mini_batch_size=2, ppo_epochs=4,
                            init_kl_coef=PPO_KL, target_kl=6.0,
                            cliprange=0.2, cliprange_value=0.2, seed=42)
        ppo_trainer = PPOTrainer(config=ppo_cfg, model=ppo_policy, ref_model=None, tokenizer=tokenizer)

        ppo_prompts = [e['prompt'] for e in pref_data[:2048]]
        ppo_history = []
        bs = ppo_cfg.batch_size
        for i in range(0, len(ppo_prompts), bs):
            chunk = ppo_prompts[i:i+bs]
            if len(chunk) < bs: break
            queries = [torch.tensor(tokenizer.encode(p, truncation=True, max_length=256)).to(device) for p in chunk]
            resp = ppo_trainer.generate(queries, return_prompt=False,
                                        max_new_tokens=64, do_sample=True, temperature=1.0,
                                        pad_token_id=PAD)
            resp_text = [tokenizer.decode(r, skip_special_tokens=True) for r in resp]
            qtext = [tokenizer.decode(q, skip_special_tokens=True) for q in queries]
            with torch.no_grad():
                rin = reward_tok([q + r for q, r in zip(qtext, resp_text)],
                                 padding=True, truncation=True, max_length=512,
                                 return_tensors='pt').to(device)
                rs = reward_model(**rin).logits.squeeze(-1)
            rewards = [torch.tensor(float(r), dtype=torch.float32) for r in rs.tolist()]
            stats = ppo_trainer.step(queries, resp, rewards)
            step = i // bs
            if step % 10 == 0:
                mr = float(np.mean([r.item() for r in rewards]))
                kl = float(stats.get('objective/kl', 0.0))
                print(f'[ppo] step={step:4d} mean_reward={mr:+.3f} kl={kl:.4f}')
                ppo_history.append({'step': step, 'mean_reward': mr, 'kl': kl,
                                    'loss_total': float(stats.get('ppo/loss/total', 0.0))})
        ppo_policy.save_pretrained(f'{PPO_DIR}/model')
        tokenizer.save_pretrained(f'{PPO_DIR}/model')
        with open(f'{PPO_DIR}/history.json', 'w') as f:
            json.dump(ppo_history, f, indent=2)
        print('PPO done.')
        del ppo_policy, reward_model; torch.cuda.empty_cache() if device == 'cuda' else None
    except Exception as e:
        print(f'[ppo] FAILED: {e}')
        print('Continuing without PPO. The two-way DPO-vs-SimPO comparison will still complete.')
        RUN_PPO = False

## 10. Generation + eval helpers (inline)

In [ ]:
@torch.no_grad()
def gen_completions(model, prompts, n=N_EVAL, max_new=64, batch=8, temp=0.7, top_p=0.95):
    model.eval()
    outs = []
    for i in range(0, min(n, len(prompts)), batch):
        chunk = prompts[i:i+batch]
        enc = tokenizer(chunk, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=True,
                             temperature=temp, top_p=top_p, pad_token_id=PAD)
        for j in range(out.shape[0]):
            pl = int(enc['attention_mask'][j].sum().item())
            resp = out[j].tolist()[pl:]
            text = tokenizer.decode(resp, skip_special_tokens=True).strip()
            outs.append({'prompt': chunk[j], 'completion': text,
                         'length': len(tokenizer.encode(text, add_special_tokens=False))})
    return outs

JUDGE_SYS = """You are an impartial judge evaluating summaries of Reddit posts. You will see a post and two candidate summaries (A and B). Choose which summary is better. Consider faithfulness, conciseness, coverage, and coherence.
Respond ONLY with a single JSON object: {"winner": "A"} or {"winner": "B"} or {"winner": "tie"}."""

def call_judge(client, post, a, b, model='gpt-4o-2024-08-06'):
    user = f'Reddit post:\n```\n{post}\n```\n\nSummary A:\n```\n{a}\n```\n\nSummary B:\n```\n{b}\n```\n\nWhich summary is better?'
    for attempt in range(4):
        try:
            r = client.chat.completions.create(model=model, temperature=0.0, max_tokens=20,
                                               response_format={'type': 'json_object'},
                                               messages=[{'role':'system','content':JUDGE_SYS},
                                                         {'role':'user','content':user}])
            text = r.choices[0].message.content
            try:
                obj = json.loads(text); w = str(obj.get('winner','')).strip().lower()
                if w in {'a','b','tie'}: return (w.upper() if w != 'tie' else 'tie'), text
            except: pass
            return 'tie', text
        except Exception as e:
            time.sleep(2**attempt)
    return 'tie', None

def pairwise(client, post, a, b):
    fwd, rf = call_judge(client, post, a, b)
    rev_raw, rr = call_judge(client, post, b, a)  # judged with B first
    rev = {'A':'B','B':'A','tie':'tie'}[rev_raw]   # flip back to A-first convention
    winner = fwd if (fwd == rev and fwd != 'tie') else 'tie'
    return {'winner': winner, 'fwd': fwd, 'rev': rev, 'raw_fwd': rf, 'raw_rev': rr}

def lc_win_rate(judgments, lens_a, lens_b):
    y = np.array([1.0 if j['winner']=='A' else 0.0 if j['winner']=='B' else 0.5 for j in judgments])
    delta = np.array([lens_a[i]-lens_b[i] for i in range(len(judgments))], dtype=float)
    std = max(1e-8, float(delta.std()))
    dn = delta/std
    b0, a = 0.0, 0.0
    for _ in range(50):
        z = b0 + a*dn; p = 1.0/(1.0+np.exp(-z))
        g0 = (p-y).mean(); ga = ((p-y)*dn).mean()
        h00 = (p*(1-p)).mean(); ha0 = (p*(1-p)*dn).mean(); haa = (p*(1-p)*dn*dn).mean()
        det = h00*haa - ha0*ha0 + 1e-8
        db = (haa*g0 - ha0*ga)/det; da = (-ha0*g0 + h00*ga)/det
        b0 -= db; a -= da
        if abs(db)+abs(da) < 1e-6: break
    return float(1.0/(1.0+math.exp(-b0))), float(y.mean()), float(a/std)

def distinct_n(samples, n=1):
    from collections import Counter
    grams = []
    for s in samples:
        toks = s.split()
        grams.extend([tuple(toks[i:i+n]) for i in range(len(toks)-n+1)])
    return (len(set(grams))/len(grams)) if grams else 0.0

def self_bleu(samples, n=4, max_pairs=200, seed=42):
    if len(samples)<2: return 0.0
    tk = [s.split() for s in samples]
    rng = random.Random(seed)
    pairs = []
    for _ in range(max_pairs):
        i = rng.randrange(len(samples))
        j = rng.randrange(len(samples)-1)
        if j>=i: j+=1
        pairs.append((i,j))
    def bleu(hyp, ref):
        if not hyp: return 0.0
        from collections import Counter
        precs=[]
        for k in range(1, n+1):
            hg=[tuple(hyp[i:i+k]) for i in range(len(hyp)-k+1)]
            rg=[tuple(ref[i:i+k]) for i in range(len(ref)-k+1)]
            if not hg: return 0.0
            rc = Counter(rg); cl=0
            for g in hg:
                if rc[g]>0: cl+=1; rc[g]-=1
            precs.append((cl+1.0)/(len(hg)+1.0))
        bp = 1.0 if len(hyp)>len(ref) else math.exp(1.0-len(ref)/max(1,len(hyp)))
        return bp * math.exp(sum(math.log(p)/n for p in precs))
    return float(np.mean([bleu(tk[i], tk[j]) for i,j in pairs]))

## 11. Run evaluation against the SFT baseline

In [ ]:
from openai import OpenAI
assert 'OPENAI_API_KEY' in os.environ, 'export OPENAI_API_KEY first'
client = OpenAI()

EVAL_DIR = f'{RESULTS}/eval'
os.makedirs(EVAL_DIR, exist_ok=True)

print('generating SFT baseline...')
sft_loaded = load_lora_model(f'{SFT_DIR}/model')
sft_gens = gen_completions(sft_loaded, eval_prompts, n=N_EVAL)
del sft_loaded; torch.cuda.empty_cache() if device == 'cuda' else None

methods = [('dpo_default', f'{DPO_DIR}/model'), ('simpo_default', f'{SIMPO_DIR}/model')]
if RUN_PPO and os.path.exists(f'{RESULTS}/ppo/default/{TS}/model'):
    methods.append(('ppo_default', f'{RESULTS}/ppo/default/{TS}/model'))

summary_rows = []
for name, ckpt in methods:
    print(f'\n=== evaluating {name} ===')
    model = load_lora_model(ckpt) if os.path.exists(os.path.join(ckpt, 'adapter_config.json')) else \
            AutoModelForCausalLM.from_pretrained(ckpt, torch_dtype=DTYPE).to(device)
    gens = gen_completions(model, eval_prompts, n=N_EVAL)
    del model; torch.cuda.empty_cache() if device == 'cuda' else None

    # Judge
    print(f'judging {N_JUDGE} pairs with GPT-4o (both orderings)...')
    judgments = []
    lens_a, lens_b = [], []
    for i in range(min(N_JUDGE, len(gens))):
        post = eval_posts[i]
        a = gens[i]['completion']
        b = sft_gens[i]['completion']
        j = pairwise(client, post, a, b)
        j['idx']=i; j['a_len']=gens[i]['length']; j['b_len']=sft_gens[i]['length']
        judgments.append(j)
        lens_a.append(gens[i]['length']); lens_b.append(sft_gens[i]['length'])
        if (i+1) % 25 == 0: print(f'  {i+1}/{N_JUDGE}')
    lcw, raww, alpha = lc_win_rate(judgments, lens_a, lens_b)

    # Diversity / length
    m_texts = [g['completion'] for g in gens]
    s_texts = [g['completion'] for g in sft_gens]
    row = {
        'name': name,
        'checkpoint': ckpt,
        'lc_win_rate': lcw,
        'raw_win_rate': raww,
        'alpha_len': alpha,
        'mean_len_method': float(np.mean([g['length'] for g in gens])),
        'mean_len_sft': float(np.mean([g['length'] for g in sft_gens])),
        'distinct_1_method': distinct_n(m_texts, 1),
        'distinct_2_method': distinct_n(m_texts, 2),
        'self_bleu_4_method': self_bleu(m_texts, 4),
        'distinct_1_sft': distinct_n(s_texts, 1),
        'distinct_2_sft': distinct_n(s_texts, 2),
        'self_bleu_4_sft': self_bleu(s_texts, 4),
        'n_judged': len(judgments),
    }
    summary_rows.append(row)
    # Save full eval json
    with open(f'{EVAL_DIR}/{name}.json', 'w') as f:
        json.dump({'name': name, 'win_rate': {'lc_win_rate':lcw, 'raw_win_rate':raww, 'alpha':alpha},
                   'method_generations': gens, 'sft_generations': sft_gens,
                   'judgments': judgments, 'row': row}, f, indent=2)
    print(f'{name}: LC win-rate = {lcw:.3f}  raw = {raww:.3f}  alpha = {alpha:+.3f}  '
          f'len_m = {row["mean_len_method"]:.1f}  len_s = {row["mean_len_sft"]:.1f}')

# Write summary CSV
import csv
with open(f'{EVAL_DIR}/summary.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    w.writeheader()
    for r in summary_rows: w.writerow(r)
print(f'\nSummary written to {EVAL_DIR}/summary.csv')

## 12. Plots

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PLOT_DIR = f'{RESULTS}/plots'
os.makedirs(PLOT_DIR, exist_ok=True)

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for hist, name, c in [(dpo_history, 'DPO', 'tab:blue'),
                       (simpo_history, 'SimPO', 'tab:orange')]:
    if not hist: continue
    steps = [h['step'] for h in hist]
    axes[0].plot(steps, [h['loss'] for h in hist], label=name, color=c)
    axes[1].plot(steps, [h['r_chosen'] for h in hist], '-',  color=c, label=f'{name} r_chosen')
    axes[1].plot(steps, [h['r_rejected'] for h in hist], '--', color=c, label=f'{name} r_rejected')
axes[0].set_xlabel('step'); axes[0].set_ylabel('loss'); axes[0].set_title('Loss'); axes[0].legend()
axes[1].set_xlabel('step'); axes[1].set_ylabel('implicit reward'); axes[1].set_title('Implicit rewards'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{PLOT_DIR}/training.png', dpi=150); plt.close()

# Summary bar chart
fig, ax = plt.subplots(figsize=(8, 4))
names = [r['name'] for r in summary_rows]
lc = [r['lc_win_rate'] for r in summary_rows]
ax.bar(names, lc); ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('LC win rate vs SFT'); ax.set_ylim(0, 1)
ax.set_title('Length-controlled win rate (higher = better)')
plt.tight_layout(); plt.savefig(f'{PLOT_DIR}/win_rates.png', dpi=150); plt.close()

# Length distributions
fig, ax = plt.subplots(figsize=(8, 4))
names = [r['name'] for r in summary_rows] + ['sft']
lens = [r['mean_len_method'] for r in summary_rows] + [summary_rows[0]['mean_len_sft']]
ax.bar(names, lens)
ax.set_ylabel('mean response length (tokens)')
ax.set_title('Output length by method')
plt.tight_layout(); plt.savefig(f'{PLOT_DIR}/lengths.png', dpi=150); plt.close()

print(f'Plots in {PLOT_DIR}')
for row in summary_rows:
    print(f"  {row['name']:15s}  lc={row['lc_win_rate']:.3f}  raw={row['raw_win_rate']:.3f}  "
          f"len={row['mean_len_method']:.1f}  d2={row['distinct_2_method']:.3f}  "
          f"sb={row['self_bleu_4_method']:.3f}")

## 13. (Optional) Bundle results for download

Zips `./results/` so you can pull it back to your laptop from the Colab file browser.


In [ ]:
import shutil
shutil.make_archive('results_bundle', 'zip', RESULTS)
print('Wrote results_bundle.zip')
print('Right-click in the Colab file browser (left pane) -> Download to pull it down.')
